# Multilingual NER Data Preprocessing

In [1]:
import os
import torch
import evaluate
from datasets import load_dataset, DatasetDict, concatenate_datasets, interleave_datasets
from transformers import AutoTokenizer, AutoModelForTokenClassification
from transformers import DataCollatorForTokenClassification

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


## Loading Datasets

In [2]:
# Hungarian
raw_hun_dataset = load_dataset("ficsort/SzegedNER", revision="convert/parquet")
# Multilingual
raw_multi_dataset = load_dataset("betterdataai/gliner-multilingual-ner-silver-v1")
# German
raw_ger_dataset = load_dataset("bltlab/open-ner-core-types", "GermEval_deu")

### Dataset Overview

#### Hungarian: `ficsort/SzegedNER`
*   **Quality**: Gold Standard (Expert Human Annotated)
*   **Description**: A subcorpus of the Szeged Treebank focusing on core NER types (PER, ORG, LOC, MISC). Annotated with high precision by linguistic experts.

#### Multilingual: `gliner-multilingual-ner-silver-v1`
*   **Quality**: Silver Standard (Machine Generated)
*   **Description**: LLM-generated dataset covering 13 languages. It is extremely granular (60+ classes).
*   **Note**: As a "Silver" dataset, it prioritizes high recall but contains machine "noise".

#### German: `bltlab/open-ner-core-types` (split: `GermEval_deu`)
*   **Quality**: Gold Standard (Expert Human Annotated)
*   **Description**: Based on the GermEval benchmark. High-quality, human-annotated German text focusing on core NER entity types.


## Inspecting Datasets

In [3]:
print(f"Features in raw Hungarian dataset: {raw_hun_dataset["train"].features.keys()}")
print(f"Features in raw multilingual dataset: {raw_multi_dataset["train"].features.keys()}")
print(f"Features in raw German dataset: {raw_ger_dataset["train"].features.keys()}")

Features in raw Hungarian dataset: dict_keys(['id', 'tokens', 'ner', 'document_id', 'sentence_id'])
Features in raw multilingual dataset: dict_keys(['id', 'language', 'source', 'tokenized_text', 'ner', 'text'])
Features in raw German dataset: dict_keys(['id', 'tokens', 'ner_tags'])


In [4]:
# Renaming 'tokenized_text' to 'tokens' for consistency
multi_dataset = raw_multi_dataset.rename_column("tokenized_text", "tokens")

# Renaming 'ner_labels' to 'ner' for consistency
ger_dataset = raw_ger_dataset.rename_column("ner_tags", "ner")

# Renaming dataset for consistency
hun_dataset = raw_hun_dataset

In [5]:
print(f"Hungarian dataset splits and sizes: {hun_dataset.num_rows}")
print(f"Multilingual dataset splits and sizes: {multi_dataset.num_rows}")
print(f"German dataset splits and sizes: {ger_dataset.num_rows}")

Hungarian dataset splits and sizes: {'train': 12282, 'validation': 940, 'test': 1728}
Multilingual dataset splits and sizes: {'train': 348958, 'validation': 19382, 'test': 19396}
German dataset splits and sizes: {'test': 5100, 'dev': 2200, 'train': 24000}


In [6]:
# Renaming 'dev' to 'validation' in the German dataset
if "dev" in ger_dataset:
    ger_dataset["validation"] = ger_dataset.pop("dev")

In [7]:
# Checking languages in multilingual dataset
print(multi_dataset["train"].unique("language"))
# Checking sources in multilingual dataset
print(multi_dataset["train"].unique("source"))

['English', 'Italian', 'Swedish', 'Vietnamese', 'Spanish', 'German', 'French', 'Chinese', 'Indonesian', 'Dutch', 'Japanese', 'Korean']
['PleIAs/SEC', 'HuggingFaceFW/finewiki', 'wikimedia/wikipedia', 'wikimedia/wikisource', 'allenai/c4', 'google/wiki40b', 'tylu0117/Bloomberg_Financial_News', 'MedRAG/pubmed', 'common-pile/pubmed', 'wikiann']


We don't need other languages apart from English from the multilingual dataset, as we already have a dataset in German and we can further filter it by only keeping records from specific sources.

## Creating English Dataset

In [8]:
def get_dataset_with_conditions(ds, source_col, lang_col, sources=None, languages=None):
    rows_to_keep = []
    for lang, src in zip(ds[lang_col], ds[source_col]):
        if (languages is not None and lang not in languages) or (sources is not None and src not in sources):
            rows_to_keep.append(False)
        else:
            rows_to_keep.append(True)
    return rows_to_keep

In [9]:
# Filtering for English language and Wikipedia sources
eng_dataset = multi_dataset.filter(
    get_dataset_with_conditions,
    batched=True, 
    fn_kwargs={
        "source_col": "source",
        "lang_col": "language",
        "sources": ["wikimedia/wikipedia", "wikiann"],
        "languages": ["English"]
        }
)

In [10]:
print(f"Hungarian dataset splits and sizes: {hun_dataset.num_rows}")
print(f"English dataset splits and sizes: {eng_dataset.num_rows}")
print(f"German dataset splits and sizes: {ger_dataset.num_rows}")

Hungarian dataset splits and sizes: {'train': 12282, 'validation': 940, 'test': 1728}
English dataset splits and sizes: {'train': 9482, 'validation': 559, 'test': 546}
German dataset splits and sizes: {'test': 5100, 'train': 24000, 'validation': 2200}


In [11]:
print(f"Total records in Hungarian dataset: {sum(hun_dataset.num_rows.values())}")
print(f"Total records in English dataset: {sum(eng_dataset.num_rows.values())}")
print(f"Total records in German dataset: {sum(ger_dataset.num_rows.values())}")

Total records in Hungarian dataset: 14950
Total records in English dataset: 10587
Total records in German dataset: 31300


Dataset sizes differ significantly, oversampling will be needed for creating the training set and train, validation and test set sizes needs to be balanced.

In [13]:
def create_balanced_splits(ds, train_ratio=0.8, val_test_ratio=0.5, seed=42):
    # Combining splits
    full_ds = concatenate_datasets([
        ds[k] for k in ds.keys()
    ])
    
    # Split into 80-20 percent
    train_temp = full_ds.train_test_split(test_size=(1 - train_ratio), seed=seed)
    
    # Split the 20 percent equally into validation and test set
    val_test = train_temp['test'].train_test_split(test_size=val_test_ratio, seed=seed)
    
    return DatasetDict({
        'train': train_temp['train'],
        'validation': val_test['train'],
        'test': val_test['test']
    })

In [14]:
hun_dataset = create_balanced_splits(hun_dataset)
eng_dataset = create_balanced_splits(eng_dataset)
ger_dataset = create_balanced_splits(ger_dataset)

# Checking the new splits
for lang, ds in [("Hunarian", hun_dataset), ("English", eng_dataset), ("German", ger_dataset)]:
    print(f"New splits for {lang} dataset: {ds.num_rows}")

New splits for Hunarian dataset: {'train': 11960, 'validation': 1495, 'test': 1495}
New splits for English dataset: {'train': 8469, 'validation': 1059, 'test': 1059}
New splits for German dataset: {'train': 25040, 'validation': 3130, 'test': 3130}


## NER Tags in Datasets

In [15]:
print("Hungarian dataset NER tags sample")
[print(row, end='\n')for row in hun_dataset["train"]['ner'][:3]]
print("\n")
print("English dataset NER tags sample")
[print(row, end='\n')for row in eng_dataset["train"]['ner'][:3]]
print("\n")
print("German dataset NER tags sample")
[print(row, end='\n')for row in ger_dataset["train"]['ner'][:3]];

Hungarian dataset NER tags sample
[0, 7, 8, 0, 0, 0, 0, 7, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
[0, 0, 0, 0, 0, 0, 0, 7, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]


English dataset NER tags sample
[[0, 0, 'full_name']]
[[0, 1, 'full_name'], [4, 6, 'date_of_birth'], [23, 23, 'city'], [26, 26, 'organization'], [57, 57, 'organization']]
[[1, 4, 'organization'], [50, 53, 'name']]


German dataset NER tags sample
[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 5, 6, 6, 0]
[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]


The English dataset uses word-span annotations while the Hungarian and German dataset uses BIO tags.

Checking granularity of NER tags in the English dataset

In [16]:
# Extracting the third tag of every row and creating a set of uniqe tags
unique_eng_tags = {
    tag[2] 
    for row in eng_dataset['train']['ner'] 
    for tag in row
}

print("English unique tags:")
print([tag for tag in sorted(unique_eng_tags)])

English unique tags:
['account_number', 'age', 'api_key', 'city', 'company', 'country', 'county', 'date', 'date_of_birth', 'date_time', 'device_id', 'education_level', 'event', 'facility', 'first_name', 'full_name', 'gender', 'job_title', 'language', 'last_name', 'law', 'local_latlng', 'medical_record_number', 'middle_name', 'name', 'national_id', 'occupation', 'organization', 'political_view', 'postal_code', 'product', 'race_ethnicity', 'state', 'street_address', 'time', 'unique_identifier', 'url', 'user_name', 'work_of_art']


In [17]:
print("Hungarian BIO tags:", hun_dataset['train'].features['ner'].feature.names)
print("German BIO tags:", ger_dataset['train'].features['ner'].feature.names)

Hungarian BIO tags: ['O', 'B-PER', 'I-PER', 'B-ORG', 'I-ORG', 'B-LOC', 'I-LOC', 'B-MISC', 'I-MISC']
German BIO tags: ['O', 'B-LOC', 'I-LOC', 'B-ORG', 'I-ORG', 'B-PER', 'I-PER']


Word-span annotation of the English dataset needs to be converted to BIO tags and Hungarian and German datasets uses different BIO tag ID's.

## Harmonizing NER Label Schemas

We must pick the schema of the Hungarian dataset as master, since we don't want to lose 'MISC' tagged words in the Hungarian text by using the German schema, and the English dataset has very granular tags which we cannot map our German and Hungarian datasets to.

In [18]:
master_label_names = hun_dataset['train'].features['ner'].feature.names
master_id2label = {i:name for i, name in enumerate(master_label_names)}
master_label2id = {name: i for i, name in enumerate(master_label_names)}

print(master_id2label)
print(master_label2id)

{0: 'O', 1: 'B-PER', 2: 'I-PER', 3: 'B-ORG', 4: 'I-ORG', 5: 'B-LOC', 6: 'I-LOC', 7: 'B-MISC', 8: 'I-MISC'}
{'O': 0, 'B-PER': 1, 'I-PER': 2, 'B-ORG': 3, 'I-ORG': 4, 'B-LOC': 5, 'I-LOC': 6, 'B-MISC': 7, 'I-MISC': 8}


Creating master dataset tags

In [19]:
label_mapping_dict = {
    # Tags pertaining to 'PER' category
    'first_name': 'PER',
    'full_name': 'PER',
    'last_name': 'PER',
    'middle_name': 'PER',
    'name': 'PER',
    'user_name': 'PER',

    # Tags pertaining to 'ORG' category
    'company': 'ORG',
    'organization': 'ORG',

    # Tags pertaining to 'LOC' category
    'city': 'LOC',
    'country': 'LOC',
    'county': 'LOC',
    'facility': 'LOC',
    'local_latlng': 'LOC',
    'postal_code': 'LOC',
    'state': 'LOC',
    'street_address': 'LOC',

    # Tags pertaining to 'MISC' category - tags which do not fit into PER/ORG/LOC. 

    'education_level': 'MISC',
    'event': 'MISC',
    'job_title': 'MISC',
    'language': 'MISC',
    'law': 'MISC',
    'occupation': 'MISC',
    'political_view': 'MISC',
    'product': 'MISC',
    'race_ethnicity': 'MISC',
    'work_of_art': 'MISC',
}

# Tags pertaining to 'O' - they are dropped
# 'account_number', 'age', 'api_key', 'date', 'date_of_birth', 'date_time', 
# 'device_id', 'gender', 'medical_record_number', 'national_id', 
# 'time', 'unique_identifier', 'url'


Converting character-span tags to BIO tags in English dataset

In [20]:
def ner_spans_to_bio_tags_batched(ds, ner_col, mapping_dict):
    standardized_ner_list = []
    
    for list_of_spans in ds[ner_col]:
        standardized_list = []
        for span in list_of_spans:
            if len(span) >= 3:
                start, end, label = span[0], span[1], span[2]
                if label in mapping_dict:
                    new_label = mapping_dict[label]
                    standardized_list.append([start, end, new_label])
                    
        standardized_ner_list.append(standardized_list)
        
    ds['ner'] = standardized_ner_list
    return ds

In [21]:
eng_dataset = eng_dataset.map(
    ner_spans_to_bio_tags_batched,
    batched=True,
    fn_kwargs={
        "ner_col": "ner",
        "mapping_dict": label_mapping_dict
    }
)

Verifying result

In [22]:
print(f"Transformed tags: {eng_dataset["train"]['ner'][:3]}")

Transformed tags: [[[0, 0, 'PER']], [[0, 1, 'PER'], [23, 23, 'LOC'], [26, 26, 'ORG'], [57, 57, 'ORG']], [[1, 4, 'ORG'], [50, 53, 'PER']]]


In [23]:
eng_dataset = eng_dataset.rename_column("ner", "ner_spans")

In [24]:
def convert_spans_to_ids_batched(ds, token_col, ner_col, master_label2id):
    all_bio_tags = []
    
    for tokens, list_of_spans in zip(ds[token_col], ds[ner_col]):
        
        # Initializing 'O' tags
        num_tokens = len(tokens)
        bio_tags = [0] * num_tokens
        
        # Mapping tags
        for span in list_of_spans:
            # Span = [start_idx, end_idx, 'label']
            start, end, label = span[0], span[1], span[2]
            
            # Inserting tags with proper prefix
            if start < num_tokens:
                bio_tags[start] = master_label2id[f"B-{label}"]
                for i in range(start + 1, min(end + 1, num_tokens)):
                    bio_tags[i] = master_label2id[f"I-{label}"]
                    
        all_bio_tags.append(bio_tags)
        
    ds['ner'] = all_bio_tags
    return ds

In [25]:
eng_dataset = eng_dataset.map(
    convert_spans_to_ids_batched,
    batched=True,
    fn_kwargs={
        "token_col":"tokens",
        "ner_col": "ner_spans",
        "master_label2id": master_label2id
        }
    )

Inspecting converted spans

In [26]:
print(eng_dataset["train"][1]['tokens'])
print(eng_dataset["train"][1]['ner_spans'])  
print(eng_dataset["train"][1]['ner'])  


['Alan', 'Virginius', '(', 'born', '3', 'January', '2003', ')', 'is', 'a', 'French', 'professional', 'footballer', 'who', 'plays', 'as', 'a', 'left', 'winger', 'for', 'Ligue', '1', 'club', 'Lille', '.', 'Career', 'Sochaux', 'On', '6', 'July', '2020', ',', 'Virginius', 'signed', 'his', 'first', 'professional', 'contract', 'with', 'Sochaux', '.', 'Virginius', 'made', 'his', 'professional', 'debut', 'with', 'Sochaux', 'in', 'a', '2', '–', '2', 'Ligue', '2', 'tie', 'with', 'Rodez']
[[0, 1, 'PER'], [23, 23, 'LOC'], [26, 26, 'ORG'], [57, 57, 'ORG']]
[1, 2, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 5, 0, 0, 3, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 3]


### Visualizing BIO Labels

In [27]:
def visualize_ner_tags(words, ner_tags, master_id2label):
    word_line = ""
    tag_line = ""

    for word, tag in zip(words, ner_tags):
        full_label = master_id2label[tag]
        max_length = max(len(word), len(full_label))

        word_line += word + " " * (max_length - len(word) + 1)
        tag_line += full_label + " " * (max_length - len(full_label) + 1)

    print(word_line)
    print(tag_line)

In [28]:
visualize_ner_tags(
    words= eng_dataset['train'][1]['tokens'],
    ner_tags= eng_dataset['train'][1]['ner'],
    master_id2label=master_id2label
)

Alan  Virginius ( born 3 January 2003 ) is a French professional footballer who plays as a left winger for Ligue 1 club Lille . Career Sochaux On 6 July 2020 , Virginius signed his first professional contract with Sochaux . Virginius made his professional debut with Sochaux in a 2 – 2 Ligue 2 tie with Rodez 
B-PER I-PER     O O    O O       O    O O  O O      O            O          O   O     O  O O    O      O   O     O O    B-LOC O O      B-ORG   O  O O    O    O O         O      O   O     O            O        O    O       O O         O    O   O            O     O    O       O  O O O O O     O O   O    B-ORG 


Creating German id mapping dictionary

In [29]:
ger_labels = ger_dataset['train'].features['ner'].feature.names
ger_id2label = {id:name for id,name in enumerate(ger_labels)}

Comparing German and Master (Hungarian) id mappings

In [30]:
print(f"German id to label mapping: {ger_id2label}")
print(f"Master id to label mapping: {master_id2label}")

German id to label mapping: {0: 'O', 1: 'B-LOC', 2: 'I-LOC', 3: 'B-ORG', 4: 'I-ORG', 5: 'B-PER', 6: 'I-PER'}
Master id to label mapping: {0: 'O', 1: 'B-PER', 2: 'I-PER', 3: 'B-ORG', 4: 'I-ORG', 5: 'B-LOC', 6: 'I-LOC', 7: 'B-MISC', 8: 'I-MISC'}


Mapping German dataset to the Hungarian ids

In [31]:
ger_dataset = ger_dataset.rename_column("ner", "ner_old_id")

In [32]:
def map_ger_tags_to_master(batch, ner_col, ger_id2label, master_label2id):
    all_new_tags = []
    for sentence_tags in batch[ner_col]:
        new_sentence_tags = [master_label2id[ger_id2label[tag_id]] for tag_id in sentence_tags]
        all_new_tags.append(new_sentence_tags)
    
    batch["ner"] = all_new_tags
    return batch

In [33]:
ger_dataset = ger_dataset.map(
    map_ger_tags_to_master,
    batched=True,
    fn_kwargs={
        "ner_col": "ner_old_id",
        "ger_id2label": ger_id2label,
        "master_label2id": master_label2id
    }
)

Verifying new mapping

In [34]:
print(f"Old German BIO mapping: {ger_dataset['train']['ner_old_id'][1]}")
print(f"New German BIO mapping: {ger_dataset['train']['ner'][1]}")

Old German BIO mapping: [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 5, 6, 6, 0]
New German BIO mapping: [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 2, 2, 0]


## Concatenating All Datasets

Keeping only necessary columns for concatenation

In [35]:
def drop_cols_in_ds(ds):
    cols_to_keep = ['tokens', 'ner']
    for split in ds.keys():
        ds[split] = ds[split].remove_columns(
            [col for col in ds[split].column_names if col not in cols_to_keep]
            )

    return ds

In [36]:
hun_dataset = drop_cols_in_ds(hun_dataset)
eng_dataset = drop_cols_in_ds(eng_dataset)
ger_dataset = drop_cols_in_ds(ger_dataset)

Checking dataset features

In [37]:
print(f"Hungarian dataset (master) feature schema: {hun_dataset['train'].features}")
print(f"English dataset feature schema: {eng_dataset['train'].features}")
print(f"German dataset feature schema: {ger_dataset['train'].features}")

Hungarian dataset (master) feature schema: {'tokens': List(Value('string')), 'ner': List(ClassLabel(names=['O', 'B-PER', 'I-PER', 'B-ORG', 'I-ORG', 'B-LOC', 'I-LOC', 'B-MISC', 'I-MISC']))}
English dataset feature schema: {'tokens': List(Value('string')), 'ner': List(Value('int64'))}
German dataset feature schema: {'tokens': List(Value('string')), 'ner': List(Value('int64'))}


In [38]:
print("Hungarian dataset NER tags sample")
[print(row, end='\n')for row in hun_dataset["train"]['ner'][:3]]
print("\n")
print("English dataset NER tags sample")
[print(row, end='\n')for row in eng_dataset["train"]['ner'][:3]]
print("\n")
print("German dataset NER tags sample")
[print(row, end='\n')for row in ger_dataset["train"]['ner'][:3]];

Hungarian dataset NER tags sample
[0, 7, 8, 0, 0, 0, 0, 7, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
[0, 0, 0, 0, 0, 0, 0, 7, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]


English dataset NER tags sample
[1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
[1, 2, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 5, 0, 0, 3, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 3]
[0, 3, 4, 4, 4, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 2, 2, 2, 0, 0, 0, 0, 0]


German dataset NER tags sample
[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 2, 2, 0]
[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]


Casting master set schema to German and English datasets

In [39]:
master_features = hun_dataset["train"].features

eng_dataset = eng_dataset.cast(master_features)
ger_dataset = ger_dataset.cast(master_features)

Verifying 'ner' column features accross datasets

In [50]:
print(f"Hungarian dataset (master) feature schema: {hun_dataset['train'].features['ner']}")
print(f"English dataset feature schema: {eng_dataset['train'].features['ner']}")
print(f"German dataset feature schema: {ger_dataset['train'].features['ner']}")

Hungarian dataset (master) feature schema: List(ClassLabel(names=['O', 'B-PER', 'I-PER', 'B-ORG', 'I-ORG', 'B-LOC', 'I-LOC', 'B-MISC', 'I-MISC']))
English dataset feature schema: List(ClassLabel(names=['O', 'B-PER', 'I-PER', 'B-ORG', 'I-ORG', 'B-LOC', 'I-LOC', 'B-MISC', 'I-MISC']))
German dataset feature schema: List(ClassLabel(names=['O', 'B-PER', 'I-PER', 'B-ORG', 'I-ORG', 'B-LOC', 'I-LOC', 'B-MISC', 'I-MISC']))


Oversampling training set

In [40]:
interleaved_dataset = DatasetDict()

In [41]:
interleaved_dataset["train"] = interleave_datasets(
    [hun_dataset["train"], eng_dataset["train"], ger_dataset["train"]],
    stopping_strategy="all_exhausted" 
)

Concatenating validation and test sets

In [42]:
interleaved_dataset["validation"] = concatenate_datasets([
    hun_dataset["validation"], 
    eng_dataset["validation"], 
    ger_dataset["validation"]
])

interleaved_dataset["test"] = concatenate_datasets([
    hun_dataset["test"], 
    eng_dataset["test"], 
    ger_dataset["test"]
])

In [43]:
print("Preprocessed dataset structure:")
print(interleaved_dataset)

Preprocessed dataset structure:
DatasetDict({
    train: Dataset({
        features: ['tokens', 'ner'],
        num_rows: 75120
    })
    validation: Dataset({
        features: ['tokens', 'ner'],
        num_rows: 5684
    })
    test: Dataset({
        features: ['tokens', 'ner'],
        num_rows: 5684
    })
})


In [44]:
print(f"Preprocessed training set size:    {len(interleaved_dataset['train'])}")
print(f"Preprocessed validation set size:  {len(interleaved_dataset['validation'])}")
print(f"Preprocessed test set size:        {len(interleaved_dataset['test'])}")

Preprocessed training set size:    75120
Preprocessed validation set size:  5684
Preprocessed test set size:        5684


Saving preprocessed dataset to disk

In [45]:
os.makedirs("../data", exist_ok=True)

In [46]:
interleaved_dataset.save_to_disk("../data/processed/harmonized_dataset/interleaved_dataset")

Saving the dataset (0/1 shards):   0%|          | 0/75120 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/5684 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/5684 [00:00<?, ? examples/s]

Saving individual datasets

In [47]:
hun_dataset.save_to_disk("../data/processed/harmonized_dataset/hun_dataset")
eng_dataset.save_to_disk("../data/processed/harmonized_dataset/eng_dataset")
ger_dataset.save_to_disk("../data/processed/harmonized_dataset/ger_dataset")

Saving the dataset (0/1 shards):   0%|          | 0/11960 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/1495 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/1495 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/8469 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/1059 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/1059 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/25040 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/3130 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/3130 [00:00<?, ? examples/s]

## Creating a Gold-Only Dataset (HU + DE)

This dataset excludes the English 'Silver' labels to test if they are degrading overall performance on the expert-annotated Hungarian and German data.

In [48]:
# Create gold-only dataset (Hungarian and German only)
gold_only_dataset = DatasetDict()

gold_only_dataset["train"] = interleave_datasets(
    [hun_dataset["train"], ger_dataset["train"]],
    stopping_strategy="all_exhausted"
)

gold_only_dataset["validation"] = concatenate_datasets([
    hun_dataset["validation"],
    ger_dataset["validation"]
])

gold_only_dataset["test"] = concatenate_datasets([
    hun_dataset["test"],
    ger_dataset["test"]
])

print("Gold-only dataset structure:")
print(gold_only_dataset)

Gold-only dataset structure:
DatasetDict({
    train: Dataset({
        features: ['tokens', 'ner'],
        num_rows: 50080
    })
    validation: Dataset({
        features: ['tokens', 'ner'],
        num_rows: 4625
    })
    test: Dataset({
        features: ['tokens', 'ner'],
        num_rows: 4625
    })
})


In [49]:
# Save gold-only dataset to disk
gold_only_dataset.save_to_disk("../data/processed/harmonized_dataset/gold_only_dataset")

Saving the dataset (0/1 shards):   0%|          | 0/50080 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/4625 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/4625 [00:00<?, ? examples/s]